In [2]:
import pandas as pd
import numpy as np

RAW = r"C:\Users\Admin\OneDrive\Documents\credit-risk-scorecard\data\raw\accepted_2007_to_2018Q4.csv"

cols = [
    "loan_amnt", "term", "installment", "emp_length", "home_ownership",
    "annual_inc", "verification_status", "purpose", "addr_state", "dti",
    "delinq_2yrs", "earliest_cr_line", "fico_range_low", "fico_range_high",
    "inq_last_6mths", "open_acc", "pub_rec", "revol_bal", "revol_util",
    "total_acc", "mort_acc", "pub_rec_bankruptcies", "application_type",
    "issue_d", "loan_status",
]
bad  = ["Charged Off", "Default", "Does not meet the credit policy. Status:Charged Off"]
good = ["Fully Paid", "Does not meet the credit policy. Status:Fully Paid"]

In [3]:
parts = []
for ch in pd.read_csv(RAW, usecols=cols, chunksize=200_000, low_memory=False):
    parts.append(ch[ch["loan_status"].isin(bad + good)])

df = pd.concat(parts, ignore_index=True)
df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y")
df["target"] = df["loan_status"].isin(bad).astype(int)
print(df.shape, df["target"].mean())

(1348099, 26) 0.19980728418313493


In [4]:
df = df[df["term"].str.strip() == "36 months"].copy()

train = df[(df["issue_d"] >= "2012-01-01") & (df["issue_d"] < "2015-01-01")]
test  = df[(df["issue_d"] >= "2015-01-01") & (df["issue_d"] < "2016-01-01")]

print("train:", train.shape, round(train["target"].mean(), 3))
print("test :", test.shape,  round(test["target"].mean(), 3))

train: (306462, 26) 0.132
test : (283026, 26) 0.149


In [5]:
df.to_parquet("../data/processed/accepted_36m.parquet")